# Variables

## `user_settings` Dictionary

This dictionary contains the main configuration parameters used by the notebook to connect to GraphDB and locate project resources.

### Fields

- **`config.folders.data_name`**: Path to the folder containing the input data files.
- **`repository_name`**: Name of the GraphDB repository to use.
- **`ont_file_name`**: File name of the ontology file.
- **`ruleset_file_name`**: File name of the reasoning ruleset.
- **`str_graphdb_url`**: URL of the GraphDB server.

### Purpose

These settings define where the data is stored and how the notebook connects to the GraphDB instance for loading ontologies, creating repositories, and building the knowledge graph.

In [ ]:
user_settings = {
    "data_folder_name": "../data/fbg_saint_antoine",
    "repository_name": "fbg_saint_antoine_test",
    "ont_file_name": "ontology.ttl",
    "ruleset_file_name": "rules.pie",
    "str_graphdb_url": "http://localhost:7200",
}

## Import libraries

In [ ]:
import os
import sys

## Importing Python Modules

This section imports various Python modules used throughout the project. The modules are organized under the `scripts` folder and are structured into utilities and graph construction modules.

In [ ]:
# ------------------------------------------------------------
# Set up the path to the 'scripts' folder for module imports
# ------------------------------------------------------------

py_code_folder_path = "../scripts"
python_code_folder = os.path.abspath(py_code_folder_path)

# `python_code_folder` should point to the 'scripts' folder
scripts_folder = os.path.abspath(python_code_folder)

# Get the parent folder of 'scripts' (the project root)
project_root = os.path.dirname(scripts_folder)

# Add the project root to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# ------------------------------------------------------------
# Import modules from the scripts package
# ------------------------------------------------------------

# Utilities
from scripts.utils import file_management as fm
from scripts.utils import time_processing as tp
from scripts.utils import configs as conf

# Graph construction modules
from scripts.graph_construction import graphdb as gd
from scripts.graph_construction.namespaces import NameSpaces
from scripts.graph_construction import multi_sources_processing as msp
from scripts.graph_construction import factoids_creation as fc
from scripts.graph_construction import fact_graph_construction as fgc

# Evaluation modules
from scripts.evaluation import create_streetnumber_factoids as csf
from scripts.evaluation import create_addr_links as cal
from scripts.evaluation import evaluate_streetnumber_versions as esv
from scripts.evaluation import evaluate_streetnumber_fragmentary as esf

### Configuration

In [ ]:
config_path = "../configs/config.yaml"
config = conf.ProjectConfig(user_settings, config_path)
np = NameSpaces()

### Creation of folders if they don't exist

In [ ]:
fm.create_folder_if_not_exists(config.folders.tmp)

### Creating the local directory in GraphDB
For the creation to work, GraphDB must be launched and therefore the URI given by `config.graphdb.base_url` must work. If the directory already exists, nothing is done.

### Options

- **`allow_removal`**: If set to `False`, the repository will not be removed during the reinitialization process. Instead, the repository will simply be emptied. This is useful in case the deletion of the repository fails, ensuring the directory is cleared without being deleted and recreated. 
- **`disable_same_as`**: When set to `True`, this option disables the use of `sameAs` in the reasoning process.

### Creation Process

The function `gd.reinitialize_repository` is used to reinitialize the repository. When called, it ensures the repository is set up according to the provided configurations (e.g., `local_config_file`, `ruleset_name`). If the repository already exists, it is reinitialized without removing it (if `allow_removal` is set to `False`).

- **`config.graphdb.base_url`**: The URL pointing to the running GraphDB instance.
- **`config.graphdb.repo_name`**: The name of the repository to be reinitialized.
- **`config.files.local_config`**: The configuration file used to initialize the repository.
- **`ruleset_name`**: The name of the ruleset to be used for reasoning, such as `"owl2-rl-optimized"` (no inference by default).
- **`allow_removal`**: Controls whether the repository can be deleted and recreated (`False` will just empty it).

For the creation of the repository to work, GraphDB must be running, and the provided `config.graphdb.base_url` must be valid. If the repository already exists, it will be reinitialized without further action.


In [ ]:
# Deleting a directory may not work, so to avoid deletion at reset time (deletion + (re)creation)
# `allow_removal` must be False, in which case the directory will just be emptied.
allow_removal = False
disable_same_as = False

gd.reinitialize_repository(config.graphdb.base_url, config.graphdb.repo_name, config.files.local_config, disable_same_as=disable_same_as, allow_removal=allow_removal) # No inference

## Loading ontologies

In [ ]:
# Loading ontology in the repository
gd.load_ontologies(config.graphdb.base_url, config.graphdb.repo_name, [config.files.ontology], config.named_graphs.ontology)
gd.add_prefixes_to_repository(config.graphdb.base_url, config.graphdb.repo_name, np.namespaces_with_prefixes)
gd.add_named_graph_prefix_to_repository(config.graphdb.base_url, config.graphdb.repo_name, "graph") # Add prefix for named graphs

## Definition of variables linked to sources

### Paris thoroughfares via Wikidata

* `wd` for "wikidata"
* `wdp_land` for "wikidata paris landmarks"
* `wdp_loc` for "wikidata paris locations"

In [ ]:
# Name of the directory where the factoid triples of Wikidata data are stored and constructed
wdp_named_graph_name = "wikidata"

# CSV file to store the result of the selection query
wdp_land_csv_file_name = "wd_paris_landmarks.csv"
wdp_land_csv_file = os.path.join(config.folders.data, wdp_land_csv_file_name)

# CSV file to store the result of the selection query
wdp_loc_csv_file_name = "wd_paris_locations.csv"
wdp_loc_csv_file = os.path.join(config.folders.data, wdp_loc_csv_file_name)

# TTL file for structuring knowledge of the Paris thoroughfares
wdp_kg_file_name = "wd_paris.ttl"
wdp_kg_file = os.path.join(config.folders.tmp, wdp_kg_file_name)

# Time interval of validity of the source (there is not end time)
wdp_valid_time = {
    "start" : {"stamp":"2024-08-26","precision":"day","calendar":"gregorian"}
    }

wdp_source = {
    "label": "Wikidata",
    "config.graphdb.lang": "mul"
}

### Nomenclature of Paris thoroughfares (Ville de Paris data)

The City of Paris data is made up of two sets:
* [names of current street rights-of-way](https://opendata.paris.fr/explore/dataset/denominations-emprises-voies-actuelles)
* [obsolete street names](https://opendata.paris.fr/explore/dataset/denominations-des-voies-caduques)

Current roads have a geometric right of way, unlike the old thoroughfares.

* `vpt` for ‘ville paris thoroughfares’
* `vpta` for ‘ville paris thoroughfares actuelles’.
* `vptc` for ‘ville paris thoroughfares caduques’.

In [ ]:
# Name of the directory where the factoid triples of Ville de Paris data are stored and constructed
vpt_named_graph_name = "ville_de_paris"

# CSV and JSON files containting data
vpta_json_file_name = "denominations-emprises-voies-actuelles.geojson"
vpta_json_file = os.path.join(config.folders.data, vpta_json_file_name)
vptc_csv_file_name = "denominations-des-voies-caduques.csv"
vptc_csv_file = os.path.join(config.folders.data, vptc_csv_file_name)

# TTL file for structuring knowledge of the Paris thoroughfares
vpt_kg_file_name = "voies_paris.ttl"
vpt_kg_file = os.path.join(config.folders.tmp, vpt_kg_file_name)

# Time interval of validity of the source (there is not end time)
vpta_valid_time = {
    "start" : {"stamp":"2025-04-01T00:00:00Z","precision":"day","calendar":"gregorian"},
    "end" : {"stamp":tp.get_current_timestamp(),"precision":"day","calendar":"gregorian"}
    }

# Description of the source for current thoroughfares of Ville de Paris
vpta_source = {
    "uri": "https://opendata.paris.fr/explore/dataset/denominations-emprises-voies-actuelles/",
    "label" : "Dénominations des emprises des voies actuelles",
    "config.graphdb.lang":"fr",
    "publisher" : {
        "label": "Département de la Topographie et de la Documentation Foncière de la Ville de Paris"
    }
}

# Description of the source for caducous thoroughfares of Ville de Paris
vptc_source = {
    "uri": "https://opendata.paris.fr/explore/dataset/denominations-des-voies-caduques/",
    "label" : "Dénominations caduques des voies",
    "config.graphdb.lang":"fr",
    "publisher" : {
        "label": "Département de la Topographie et de la Documentation Foncière de la Ville de Paris"
    }
}

### Base Adresse Nationale (BAN)

Data from the [Base Adresse Nationale (BAN)](https://adresse.data.gouv.fr/base-adresse-nationale) (National Address Base), available [here](https://adresse.data.gouv.fr/data/ban/adresses/latest/csv)

bpa` for ‘BAN paris addresses’

In [ ]:
# Name of the directory where the factoid triples of BAN data are stored and constructed
bpa_named_graph_name = "ban_adresses"

# CSV file containting data
bpa_csv_file_name = "ban_adresses.csv"
bpa_csv_file = os.path.join(config.folders.data, bpa_csv_file_name)

# TTL file for structuring knowledge of Paris addresses
bpa_kg_file_name = "ban_adresses.ttl"
bpa_kg_file = os.path.join(config.folders.tmp, bpa_kg_file_name)

# Time interval of validity of the source (there is not end time)
bpa_valid_time = {
    "start" : {"stamp":"2024-01-01","precision":"day","calendar":"gregorian"},
    "end" : {"stamp":"2025-01-01","precision":"day","calendar":"gregorian"}
    }

bpa_source = {
    "label" : "Base Adresse Nationale",
    "config.graphdb.lang":"fr",
    "publisher" : {
        "label": "DINUM / ANCT / IGN"
    }
}

### OpenStreetMap (OSM)

Extracting data from OpenStreetMap

In [ ]:
# Name of the directory where the factoid triples of OSM data are stored and constructed
osm_named_graph_name = "osm"

# CSV files containting data
osm_csv_file_name = "osm_adresses.csv"
osm_csv_file = os.path.join(config.folders.data, osm_csv_file_name)
osm_hn_csv_file_name = "osm_hn_adresses.csv"
osm_hn_csv_file = os.path.join(config.folders.data, osm_hn_csv_file_name)

# TTL file for structuring knowledge of OSM addresses
osm_kg_file_name = "osm_adresses.ttl"
osm_kg_file = os.path.join(config.folders.tmp, osm_kg_file_name)

# Time interval of validity of the source (there is not end time)
osm_valid_time = {
    "start" : {"stamp":"2025-05-01","precision":"day","calendar":"gregorian"},
    "end" : {"stamp":tp.get_current_timestamp(),"precision":"day","calendar":"gregorian"}
    }

osm_source = {
    "label" : "OpenStreetMap",
    "config.graphdb.lang":"mul"
}

### Integration of data from Geojson files describing thoroughfares

These datasets are derived from the vectorisation of several historical maps of Paris, including:
- the Plan Delagrive (1728);
- the Verniquet atlas (1784–1791);
- the Vasserot atlas (1810–1836);
- Andriveau’s plan (1849);
- the Municipal Atlas map (1888).


#### Plan Delagrive 1728

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
del_1728_th_named_graph_name = "plan_delagrive_1728_voies"

# Geojson file containting data
del_1728_th_geojson_file_name = "plan_delagrive_1728_voies.geojson"
del_1728_th_geojson_file = os.path.join(config.folders.data, del_1728_th_geojson_file_name)
del_1728_th_kg_file_name = "plan_delagrive_1728_voies.ttl"
del_1728_th_kg_file = os.path.join(config.folders.tmp, del_1728_th_kg_file_name)

del_1728_th_name_attribute = "streetname"

# Description of the source within a dictionary
del_1728_th_source = {
    "lang" : "fr", 
    "uri" : "https://gallica.bnf.fr/ark:/12148/btv1b53085122h",
    "label" : "Neuvieme Plan de Paris. Ses accroissemens sous le Regne de Louis XV[...] ",
    "publisher" : {
        "label": "Delagrive"
        }
}

# Time interval of validity of the source
del_1728_th_valid_time = {
    "start" : {"stamp":"1827-01-01","precision":"year","calendar":"gregorian"},
    "end" : {"stamp":"1829-01-01","precision":"year","calendar":"gregorian"},
}

#### Verniquet atlas

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
ve_1790_th_named_graph_name = "atlas_verniquet_1791_voies"

# Geojson file containting data
ve_1790_th_geojson_file_name = "atlas_verniquet_1791_voies.geojson"
ve_1790_th_geojson_file = os.path.join(config.folders.data, ve_1790_th_geojson_file_name)
ve_1790_th_kg_file_name = "atlas_verniquet_1791_voies.ttl"
ve_1790_th_kg_file = os.path.join(config.folders.tmp, ve_1790_th_kg_file_name)

ve_1790_th_name_attribute = "streetname"

# Description of the source within a dictionary
ve_1790_th_source = {
    "uri": "https://gallica.bnf.fr/ark:/12148/bpt6k3167995",
    "lang" : "fr", 
    "label" : "Atlas Général de la Ville",
    "publisher" : {
        "label": "Verniquet"
        }
}

# Time interval of validity of the source
ve_1790_th_valid_time = {
    "start" : {"stamp":"1784-01-01","precision":"year","calendar":"gregorian"},
    "end" : {"stamp":"1791-01-01","precision":"year","calendar":"gregorian"},
}

#### Vasserot atlas

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
va_1810_th_named_graph_name = "atlas_vasserot_1810_voies"

# Geojson file containting data
va_1810_th_geojson_file_name = "atlas_vasserot_1810_voies.geojson"
va_1810_th_geojson_file = os.path.join(config.folders.data, va_1810_th_geojson_file_name)
va_1810_th_kg_file_name = "atlas_vasserot_1810_voies.ttl"
va_1810_th_kg_file = os.path.join(config.folders.tmp, va_1810_th_kg_file_name)

va_1810_th_name_attribute = "nom_entier"

# Description of the source within a dictionary
va_1810_th_source = {
    "uri": "www.fabriquenumeriquedupasse.fr/explore/dataset/alpage-voies-vasserot",
    "lang" : "fr", 
    "label" : "Cadastre de Paris par îlot : 1810-1836",
    "publisher" : {
        "label": "Vasserot"
        }
}

# Time interval of validity of the source
va_1810_th_valid_time = {
    "start" : {"stamp":"1810-01-01","precision":"year","calendar":"gregorian"},
    "end" : {"stamp":"1836-01-01","precision":"year","calendar":"gregorian"},
}

#### 1836 Jacoubet Atlas streets

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
ja_1836_th_named_graph_name = "atlas_jacoubet_1836_voies"

# Geojson file containting data
ja_1836_th_geojson_file_name = "atlas_jacoubet_1836_voies.geojson"
ja_1836_th_geojson_file = os.path.join(config.folders.data, ja_1836_th_geojson_file_name)
ja_1836_th_kg_file_name = "atlas_jacoubet_1836_voies.ttl"
ja_1836_th_kg_file = os.path.join(config.folders.tmp, ja_1836_th_kg_file_name)

ja_1836_th_name_attribute = "nom_entier"

# Description of the source within a dictionary
ja_1836_th_source = {
    "uri":"https://bibliotheques-specialisees.paris.fr/ark:/73873/pf0000212158",
    "lang" : "fr", 
    "label" : "Atlas général de la ville, des faubourgs et des monuments de Paris",
    "publisher" : {
        "label": "Jacoubet"
        }
}

# Time interval of validity of the source
ja_1836_th_valid_time = {
    "start" : {"stamp":"1836-01-01","precision":"year","calendar":"gregorian"},
    "end" : {"stamp":"1838-01-01","precision":"year","calendar":"gregorian"},
}

#### Andriveau atlas

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
an_1849_th_named_graph_name = "plan_andriveau_1849_voies"

# Geojson file containting data
an_1849_th_geojson_file_name = "plan_andriveau_1849_voies.geojson"
an_1849_th_geojson_file = os.path.join(config.folders.data, an_1849_th_geojson_file_name)
an_1849_th_kg_file_name = "plan_andriveau_1849_voies.ttl"
an_1849_th_kg_file = os.path.join(config.folders.tmp, an_1849_th_kg_file_name)

an_1849_th_name_attribute = "streetname"

# Description of the source within a dictionary
an_1849_th_source = {
    "lang" : "fr", 
    "label" : "Plan de Paris comprenant l'enceinte des fortifications",
    "publisher" : {
        "label": "Andriveau"
        }
}

# Time interval of validity of the source
an_1849_th_valid_time = {
    "start" : {"stamp":"1848-01-01","precision":"year","calendar":"gregorian"},
    "end" : {"stamp":"1850-01-01","precision":"year","calendar":"gregorian"},
}

#### 1888 Municipal Atlas of Paris

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
am_1888_th_named_graph_name = "atlas_municipal_1888_voies"

# Geojson file containting data
am_1888_th_geojson_file_name = "atlas_municipal_1888_voies.geojson"
am_1888_th_geojson_file = os.path.join(config.folders.data, am_1888_th_geojson_file_name)
am_1888_th_kg_file_name = "atlas_municipal_1888_voies.ttl"
am_1888_th_kg_file = os.path.join(config.folders.tmp, am_1888_th_kg_file_name)

am_1888_th_name_attribute = "streetname"

# Description of the source within a dictionary
am_1888_th_source = {
    "uri":"https://bibliotheques-specialisees.paris.fr/ark:/73873/pf0000935116",
    "lang" : "fr", 
    "label" : "Plan de l'atlas municipal de 1888",
    "publisher" : {
        "label": "Poubelle"
        }
}

# Time interval of validity of the source
am_1888_th_valid_time = {
    "start" : {"stamp":"1887-01-01","precision":"year","calendar":"gregorian"},
    "end" : {"stamp":"1889-01-01","precision":"year","calendar":"gregorian"},
}

### Integration of data from Geojson files describing street numbers

The integration of data from GeoJSON files describing street numbers is based on the vectorisation of several historical sources:
- the General cadastre of Paris (1807);
- the Vasserot atlas;
- the Jacoubet atlas;
- the Municipal Atlas map (1888).


#### 1807 General cadastre of Paris

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
cad_1807_addr_named_graph_name = "cadastre_paris_1807_adresses"

# Geojson file containting data
cad_1807_addr_geojson_file_name = "cadastre_paris_1807_adresses.geojson"
cad_1807_addr_geojson_file = os.path.join(config.folders.data, cad_1807_addr_geojson_file_name)
cad_1807_addr_kg_file_name = "cadastre_paris_1807_adresses.ttl"
cad_1807_addr_kg_file = os.path.join(config.folders.tmp, cad_1807_addr_kg_file_name)

# Attribute names of the street number value and the street name
cad_1807_addr_sn_name_property = "NUMERO TXT"
cad_1807_addr_th_name_property = "NOM_SAISI"

# Description of the source within a dictionary
cad_1807_addr_source = {
    "lang" : "fr", 
    "label" : "Adresses du cadastre général de Paris de 1807",
    "publisher" : {
        "label": "Ville de Paris"
        }
}

# Time interval of validity of the source
cad_1807_addr_valid_time = {
    "start" : {"stamp":"1806-01-01","precision":"year","calendar":"gregorian"},
    "end" : {"stamp":"1808-01-01","precision":"year","calendar":"gregorian"},
}

#### Vasserot atlas

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
va_1810_addr_named_graph_name = "atlas_vasserot_1810_adresses"

# Geojson file containting data
va_1810_addr_geojson_file_name = "atlas_vasserot_1810_adresses.geojson"
va_1810_addr_geojson_file = os.path.join(config.folders.data, va_1810_addr_geojson_file_name)
va_1810_addr_kg_file_name = "atlas_vasserot_1810_adresses.ttl"
va_1810_addr_kg_file = os.path.join(config.folders.tmp, va_1810_addr_kg_file_name)

# Attribute names of the street number value and the street name
va_1810_addr_sn_name_property = "num_voies"
va_1810_addr_th_name_property = "nom_entier"

# Description of the source within a dictionary
va_1810_addr_source = {
    "uri": "https://www.fabriquenumeriquedupasse.fr/explore/dataset/alpage-adresses-vasserot",
    "lang" : "fr", 
    "label" : "Cadastre de Paris par îlot : 1810-1836",
    "publisher" : {
        "label": "Vasserot"
        }
}

# Time interval of validity of the source
va_1810_addr_valid_time = va_1810_th_valid_time

#### 1836 Jacoubet Atlas

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
ja_1836_addr_named_graph_name = "atlas_jacoubet_1836_adresses"

# Geojson file containting data
ja_1836_addr_geojson_file_name = "atlas_jacoubet_1836_adresses.geojson"
ja_1836_addr_geojson_file = os.path.join(config.folders.data, ja_1836_addr_geojson_file_name)
ja_1836_addr_kg_file_name = "atlas_jacoubet_1836_adresses.ttl"
ja_1836_addr_kg_file = os.path.join(config.folders.tmp, ja_1836_addr_kg_file_name)

# Attribute names of the street number value and the street name
ja_1836_addr_sn_name_property = "num_voies"
ja_1836_addr_th_name_property = "nom_entier"

# Description of the source within a dictionary
ja_1836_addr_source = ja_1836_th_source

# Time interval of validity of the source
ja_1836_addr_valid_time = ja_1836_th_valid_time

#### 1888 Municipal Atlas of Paris

In [ ]:
# Name of the directory where data factoid triples are stored and constructed
am_1888_addr_named_graph_name = "atlas_municipal_1888_adresses"

# Geojson file containting data
am_1888_addr_geojson_file_name = "atlas_municipal_1888_adresses.geojson"
am_1888_addr_geojson_file = os.path.join(config.folders.data, am_1888_addr_geojson_file_name)
am_1888_addr_kg_file_name = "atlas_municipal_1888_adresses.ttl"
am_1888_addr_kg_file = os.path.join(config.folders.tmp, am_1888_addr_kg_file_name)

# Attribute names of the street number value and the street name
am_1888_addr_sn_name_property = "numbers_va"
am_1888_addr_th_name_property = "normalised"

# Description of the source within a dictionary
am_1888_addr_source = am_1888_th_source

# Time interval of validity of the source
am_1888_addr_valid_time = am_1888_th_valid_time

### Events

TTL file describing events

In [ ]:
# Name of the directory where the factoid triples of events data are stored and constructed
events_named_graph_name = "source_events"

# Event file containting data
events_json_file_name = "events.json"
events_json_file = os.path.join(config.folders.data, events_json_file_name)

# Final TTL file of factoids from events
events_kg_file_name = "events.ttl"
events_kg_file = os.path.join(config.folders.tmp, events_kg_file_name)

## Final and iterative process

### Create source graphs

For each source, factoids are created independently in separate named graphs

In [ ]:
# Process for Ville de Paris
g = fc.create_graph_from_ville_paris(vpta_json_file, vptc_csv_file, vpta_valid_time, vpta_source, vptc_source, "fr", vpa_file_format="json", vpc_file_format="csv")
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, vpt_named_graph_name, vpt_kg_file, "source", config.named_graphs.metadata)

# Process for BAN
g = fc.create_graph_from_paris_ban(bpa_csv_file, bpa_valid_time, bpa_source, "fr")
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, bpa_named_graph_name, bpa_kg_file, "source", config.named_graphs.metadata)

# Process for Wikidata
# fc.get_data_from_wikidata(wdp_land_csv_file, wdp_loc_csv_file)
g = fc.create_graph_from_wikidata(wdp_land_csv_file, wdp_loc_csv_file, wdp_source, config.graphdb.lang)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, wdp_named_graph_name, wdp_kg_file, "source", config.named_graphs.metadata)

# Process for OpenStreetMap
g = fc.create_graph_from_osm(osm_csv_file, osm_hn_csv_file, osm_valid_time, osm_source, config.graphdb.lang)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, osm_named_graph_name, osm_kg_file, "source", config.named_graphs.metadata)

# Process for streets of the Delagrive map
g = fc.create_graph_from_geojson_states_of_thoroughfares(
    del_1728_th_geojson_file, config.graphdb.lang, del_1728_th_valid_time, del_1728_th_source,
    del_1728_th_name_attribute, del_1728_th_name_attribute)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, del_1728_th_named_graph_name, del_1728_th_kg_file, "source", config.named_graphs.metadata)

# Process for streets of the Verniquet Atlas
g = fc.create_graph_from_geojson_states_of_thoroughfares(
    ve_1790_th_geojson_file, config.graphdb.lang, ve_1790_th_valid_time, ve_1790_th_source,
    ve_1790_th_name_attribute, ve_1790_th_name_attribute)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, ve_1790_th_named_graph_name, ve_1790_th_kg_file, "source", config.named_graphs.metadata)

# Process for streets of the Vasserot Atlas
g = fc.create_graph_from_geojson_states_of_thoroughfares(
    va_1810_th_geojson_file, config.graphdb.lang, va_1810_th_valid_time, va_1810_th_source,
    va_1810_th_name_attribute, va_1810_th_name_attribute)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, va_1810_th_named_graph_name, va_1810_th_kg_file, "source", config.named_graphs.metadata)

# Process for streets of the 1836 Jacoubet Atlas
g = fc.create_graph_from_geojson_states_of_thoroughfares(
    ja_1836_th_geojson_file, config.graphdb.lang, ja_1836_th_valid_time, ja_1836_th_source,
    ja_1836_th_name_attribute, ja_1836_th_name_attribute)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, ja_1836_th_named_graph_name, ja_1836_th_kg_file, "source", config.named_graphs.metadata)

# Process for streets of the Andriveau map
g = fc.create_graph_from_geojson_states_of_thoroughfares(
    an_1849_th_geojson_file, config.graphdb.lang, an_1849_th_valid_time, an_1849_th_source,
    an_1849_th_name_attribute, an_1849_th_name_attribute)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, an_1849_th_named_graph_name, an_1849_th_kg_file, "source", config.named_graphs.metadata)

# Process for streets of the 1888 Municipal Atlas
g = fc.create_graph_from_geojson_states_of_thoroughfares(
    am_1888_th_geojson_file, config.graphdb.lang, am_1888_th_valid_time, am_1888_th_source,
    am_1888_th_name_attribute, am_1888_th_name_attribute)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, am_1888_th_named_graph_name, am_1888_th_kg_file, "source", config.named_graphs.metadata)

# Process for adresses of the General Cadastre of Paris
g = fc.create_graph_from_geojson_states_of_streetnumbers(
    cad_1807_addr_geojson_file, config.graphdb.lang, cad_1807_addr_valid_time, cad_1807_addr_source,
    cad_1807_addr_sn_name_property, cad_1807_addr_th_name_property)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, cad_1807_addr_named_graph_name, cad_1807_addr_kg_file, "source", config.named_graphs.metadata)

# Process for adresses of the Vasserot Atlas
g = fc.create_graph_from_geojson_states_of_streetnumbers(
    va_1810_addr_geojson_file, config.graphdb.lang, va_1810_addr_valid_time, va_1810_addr_source,
    va_1810_addr_sn_name_property, va_1810_addr_th_name_property)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, va_1810_addr_named_graph_name, va_1810_addr_kg_file, "source", config.named_graphs.metadata)

# Process for adresses of the 1836 Jacoubet Atlas
g = fc.create_graph_from_geojson_states_of_streetnumbers(
    ja_1836_addr_geojson_file, config.graphdb.lang, ja_1836_addr_valid_time, ja_1836_addr_source,
    ja_1836_addr_sn_name_property, ja_1836_addr_th_name_property)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, ja_1836_addr_named_graph_name, ja_1836_addr_kg_file, "source", config.named_graphs.metadata)

# Process for adresses of the 1888 Municipal Atlas plan
g = fc.create_graph_from_geojson_states_of_streetnumbers(
    am_1888_addr_geojson_file, config.graphdb.lang, am_1888_addr_valid_time, am_1888_addr_source,
    am_1888_addr_sn_name_property, am_1888_addr_th_name_property)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, am_1888_addr_named_graph_name, am_1888_addr_kg_file, "source", config.named_graphs.metadata)

# Process for events
g = fc.create_graph_from_events(events_json_file)
msp.transfert_rdflib_graph_to_named_graph_repository(g, config.graphdb.base_url, config.graphdb.repo_name, events_named_graph_name, events_kg_file)

### Creation of the fact graph from source ones

#### Remove facts and inter_source named graph before importing factoids

In [ ]:
gd.remove_named_graph(config.graphdb.base_url, config.graphdb.repo_name, config.named_graphs.facts)
msp.remove_construction_named_graphs(config.graphdb.base_url, config.graphdb.repo_name)

#### Add definition of facts and construction named graph in meta named graph

In [ ]:
construction_named_graphs = [config.named_graphs.inter_sources, config.named_graphs.comparisons, config.named_graphs.labels]

for graph in construction_named_graphs:
    msp.add_construction_named_graph_to_repository(config.graphdb.base_url, config.graphdb.repo_name, config.named_graphs.metadata, graph)

#### Build facts graph as a final one

In [ ]:
fgc.build_fact_graph_from_sources(
    config.graphdb.base_url, config.graphdb.repo_name, config.named_graphs.facts, config.graphdb.facts_label,
    config.named_graphs.metadata, config.named_graphs.inter_sources,
    config.named_graphs.labels, config.files.labels,
    config.named_graphs.temporary, config.named_graphs.comparisons, config.files.comparisons, config.comparison_settings,
    lang=config.graphdb.lang
)

#### Get fragmentary data
After having created facts graph, the following step tends to create synthetic fragmentary which will be inserted in the graph :
* states
* events

Then, two new fact graphs will be created : one by adding only states and an other one by adding states and events

In [ ]:
fragmentary_data_folder_name = "fragmentary_data"
fragmentary_data_folder = os.path.join(config.folders.tmp, fragmentary_data_folder_name)
fm.create_folder_if_not_exists(fragmentary_data_folder)

# Sampling ratios for evaluation of fragmentary descriptions
version_sample_ratio=0.6
change_sample_ratio=0.6

csf.create_streetnumber_fragmentary_descriptions(
    config.graphdb.base_url, config.graphdb.repo_name, config.named_graphs.facts,
    fragmentary_data_folder, fragmentary_data_folder, config.comparison_settings,
    version_sample_ratio, change_sample_ratio)

### Fragmentary streetnumbers

JSON file describing states of fragmentary street numbers

In [ ]:
# Name of the directory where the factoid triples of fragmentary street numbers data are stored and constructed
fragmentary_sn_states_named_graph_name = "fragmentary_states_streetnumbers"
fragmentary_sn_states_named_graph_label = "Faits construits en ajoutant des états fragmentaires de numéros de rue"

# fragmentary street numbers file containting data
fragmentary_sn_states_json_file_name = "fragmentary_states_streetnumbers.json"
fragmentary_sn_states_json_file = os.path.join(fragmentary_data_folder, fragmentary_sn_states_json_file_name)

# Final TTL file of factoids from states of fragmentary street numbers
fragmentary_sn_states_kg_file_name = "fragmentary_states_streetnumbers.ttl"
fragmentary_sn_states_kg_file = os.path.join(config.folders.tmp, fragmentary_sn_states_kg_file_name)

JSON file describing events of fragmentary street numbers

In [ ]:
# Name of the directory where the factoid triples of fragmentary street numbers data are stored and constructed
fragmentary_sn_events_named_graph_name = "fragmentary_events_streetnumbers"
fragmentary_sn_events_named_graph_name_label = "Faits construits en ajoutant des états et des événements fragmentaires de numéros de rue"

# fragmentary street numbers file containting data
fragmentary_sn_events_json_file_name = "fragmentary_events_streetnumbers.json"
fragmentary_sn_events_json_file = os.path.join(fragmentary_data_folder, fragmentary_sn_events_json_file_name)

# Final TTL file of factoids from states of fragmentary street numbers
fragmentary_sn_events_kg_file_name = "fragmentary_events_streetnumbers.ttl"
fragmentary_sn_events_kg_file = os.path.join(config.folders.tmp, fragmentary_sn_events_kg_file_name)

Add fragmentary data in the repository in two named graphs (one for states, the other for events)

In [ ]:
# Process for fragmentary states streetnumbers
g = fc.create_graph_from_states(fragmentary_sn_states_json_file)
msp.transfert_rdflib_graph_to_named_graph_repository(
    g, config.graphdb.base_url, config.graphdb.repo_name, fragmentary_sn_states_named_graph_name, fragmentary_sn_states_kg_file, "source", config.named_graphs.metadata)

# Process for fragmentary events streetnumbers (is_active=False to not activate events for the moment)
g = fc.create_graph_from_events(fragmentary_sn_events_json_file)
msp.transfert_rdflib_graph_to_named_graph_repository(
    g, config.graphdb.base_url, config.graphdb.repo_name, fragmentary_sn_events_named_graph_name, fragmentary_sn_events_kg_file, "source", config.named_graphs.metadata, is_active=False)

Create a new facts graph by adding only states fragmentary data

In [ ]:
frag_sn_states_facts_named_graph_name = "facts_with_fragmentary_sn_states"
frag_sn_states_facts_named_graph_label = "Faits avec des états fragmentaires de numéros de rue"

fgc.build_fact_graph_from_sources(
    config.graphdb.base_url, config.graphdb.repo_name,
    frag_sn_states_facts_named_graph_name, frag_sn_states_facts_named_graph_label,
    config.named_graphs.metadata, config.named_graphs.inter_sources,
    config.named_graphs.labels, config.files.labels,
    config.named_graphs.temporary, config.named_graphs.comparisons, config.files.comparisons, config.comparison_settings,
    lang=config.graphdb.lang
)

Create a new facts graph by adding states and events fragmentary data

In [ ]:
frag_sn_states_events_facts_named_graph_name = "facts_with_fragmentary_sn_states_and_events"
frag_sn_states_events_facts_named_graph_label = "Faits avec des états et des événements fragmentaires de numéros de rue"

# Activate fragmentary events streetnumbers named graph in the repository before building new facts graph
msp.set_named_graph_active(config.graphdb.base_url, config.graphdb.repo_name, fragmentary_sn_events_named_graph_name, config.named_graphs.metadata, active=True)

fgc.build_fact_graph_from_sources(
    config.graphdb.base_url, config.graphdb.repo_name,
    frag_sn_states_events_facts_named_graph_name, frag_sn_states_events_facts_named_graph_label,
    config.named_graphs.metadata, config.named_graphs.inter_sources,
    config.named_graphs.labels, config.files.labels,
    config.named_graphs.temporary,
    config.named_graphs.comparisons, config.files.comparisons, config.comparison_settings,
    lang=config.graphdb.lang
)

# Evaluation

## Variables

In [ ]:
# Variables
links_folder_name = "links"
db_config_file = "../configs/db_config.ini"
proj_config_file = "../configs/project_config.ini"

# Create links folder if it does not exist
links_folder = os.path.join(config.folders.tmp, links_folder_name) 
fm.create_folder_if_not_exists(links_folder)

# Output file paths for ground truth and unmatched street numbers
links_ground_truth = os.path.join(links_folder, "links_ground_truth.csv")
sn_without_link_ground_truth = os.path.join(links_folder, "sn_without_link_ground_truth.csv")

source_mapping = {
    "cadastre_paris_1807_adresses": {"order": 1, "label": cad_1807_addr_source.get("label")},
    "atlas_vasserot_1810_adresses": {"order": 2, "label": va_1810_addr_source.get("label")},
    "atlas_jacoubet_1836_adresses": {"order": 3, "label": ja_1836_addr_source.get("label")},
    "atlas_municipal_1888_adresses": {"order": 4, "label": am_1888_addr_source.get("label")},
    "ban_adresses": {"order": 5, "label": bpa_source.get("label")},
    "osm_adresses": {"order": 6, "label": osm_source.get("label")},
    }

# Settings for each historical address source (GeoJSON)
sources_settings = [
    {
        'source_name': 'cadastre_paris_1807_adresses',
        'file': cad_1807_addr_geojson_file,
        'number_prop': cad_1807_addr_sn_name_property,
        'street_name_prop': cad_1807_addr_th_name_property,
        'epsg_code': 2154
    },
    {
        'source_name': 'atlas_vasserot_1810_adresses',
        'file': va_1810_addr_geojson_file,
        'number_prop': va_1810_addr_sn_name_property,
        'street_name_prop': va_1810_addr_th_name_property,
        'epsg_code': 4326
    },
    {
        'source_name': 'atlas_jacoubet_1836_adresses',
        'file': ja_1836_addr_geojson_file,
        'number_prop': ja_1836_addr_sn_name_property,
        'street_name_prop': ja_1836_addr_th_name_property,
        'epsg_code': 2154
    },
    {
        'source_name': 'atlas_municipal_1888_adresses',
        'file': am_1888_addr_geojson_file,
        'number_prop': am_1888_addr_sn_name_property,
        'street_name_prop': am_1888_addr_th_name_property,
        'epsg_code': 2154
    }
]

# Settings for BAN (Base Adresse Nationale) CSV source
ban_settings = {
    'source_name': 'ban_adresses',
    'file': bpa_csv_file,
    'number_prop': 'numero',
    'repetition_prop': 'rep',
    'street_name_prop': 'nom_voie',
    'lat_prop': 'lat',
    'lon_prop': 'lon',
    'epsg_code': 4326
}

# Settings for OSM (OpenStreetMap) CSV sources
osm_settings = {
    'source_name': 'osm_adresses',
    'file': osm_csv_file,
    'hn_file': osm_hn_csv_file,
    'join_prop': 'houseNumberId',
    'number_prop': 'houseNumberLabel',
    'street_name_prop': 'streetName',
    'geom_prop': 'houseNumberGeomWKT',
    'epsg_code': 4326
}

## Create links

From the various address sources (GeoJSON and CSV), create links between observations of the same street number across two consecutive sources. For each link, indicate whether the geometries of the corresponding numbers are similar, using a maximum distance threshold defined in `proj_config_file`.

In [ ]:
cal.create_links(
    db_config_file, proj_config_file,
    sources_settings, ban_settings, osm_settings, source_mapping, links_folder
)

### Version Quality Metrics Explanation

The evaluation produces two sets of metrics for the reconstructed street number versions:

1. **Number of Versions Metric**  
   - Measures whether the reconstructed street number (SN) has the **same number of versions** as the reference (ground-truth) data.  
   - For each SN:
     - `true` → the number of versions matches the reference.  
     - `false` → the number of versions does not match.  
   - `total` → total number of street numbers evaluated.  
   - `IoU` (Intersection over Union) → fraction of street numbers with the correct number of versions.

2. **Sources Metric**  
   - Measures whether the **sources associated with each version** match the reference, ignoring a specified fragmentary source label.  
   - For each SN:
     - `true` → all versions have matching sources.  
     - `false` → at least one version has mismatched sources.  
   - `total` → total number of street numbers evaluated.  
   - `IoU` → fraction of street numbers with correctly matching sources.

These metrics help assess how well the reconstructed versions replicate both the **temporal granularity** (number of versions) and the **source provenance** of the original data.


In [ ]:
version_quality_metrics = esv.run_version_evaluation(config.graphdb.base_url, config.graphdb.repo_name, config.named_graphs.facts, source_mapping, config.folders.data, links_folder)

# Evaluation of the Impact of Fragmentary Knowledge Insertion

This section analyses the impact of inserting fragmentary states and events
into the knowledge graph on the reconstructed evolution of street numbers.
The evaluation focuses on two complementary dimensions:
(1) attribute version stability and
(2) temporal stability of detected changes.

---

## 1. Attribute Version Stability

### Objective

The first evaluation aims to verify whether the insertion of fragmentary
states and events alters the composition of geometry attribute versions.
More precisely, we check whether versions resulting from the merging of
the same set of sources remain unchanged after enrichment.

### Method

For each street number, we compare:
- the geometry attribute versions extracted from the reference graph,
- the versions extracted from graphs enriched with:
  - fragmentary states only,
  - fragmentary states and events.

Versions whose provenance includes fragmentary factoids are identified
using the source label  
**“Factoïdes générés pour les numéros de rue”**.

The evaluation distinguishes between:
- *unchanged versions*: versions identical to the reference,
- *modified versions*: versions whose source composition differs.

### Results

The results show that:

- In the **fragmentary states** configuration, the vast majority of geometry
  versions remain unchanged, indicating that the insertion of intermediate
  states does not affect the final merged representations.
- In the **fragmentary states and events** configuration, a similar behaviour
  is observed, with only a limited number of modified versions.

These results indicate that the enrichment process preserves the stability
of attribute version reconstruction, even when fragmentary knowledge is added.

---

## 2. Temporal Stability of Attribute Changes

### Objective

The second evaluation assesses whether inserting fragmentary states and events
modifies the temporal localisation of attribute changes.

The objective is to ensure that detected change instants remain consistent
with those obtained from the reference graph.

### Method

For each street number, we compare the change times extracted from:
- the reference graph,
- the enriched graphs.

Each change is classified as:
- *identical*: detected at the same time as in the reference graph,
- *shifted*: detected but at a different time,
- *missing*: present in the reference but absent in the enriched graph.

### Results

The evaluation highlights that:

- Most changes detected in the reference graph are recovered at the same
  temporal positions after enrichment.
- Only a small number of changes are temporally shifted or missing,
  primarily due to the increased temporal granularity introduced by
  fragmentary states.

Overall, the results confirm that the insertion of fragmentary knowledge
does not significantly distort the temporal structure of attribute evolution.

---

## 3. Discussion

These evaluations demonstrate that the proposed enrichment strategy
introduces additional temporal detail without compromising the coherence
of reconstructed evolutions.

In particular:
- attribute versions remain stable despite the insertion of intermediate states,
- change detection remains temporally consistent.

This confirms the robustness of the PeGazUs approach when integrating
heterogeneous and fragmentary historical data.


In [ ]:
# Detect street numbers that has been modified during the process
frag_source_label = "Factoïdes générés pour les numéros de rue"

fragmentary_evaluation_metrics = esf.run_fragmentary_evaluation(
    links_folder,
    config.graphdb.base_url,
    config.graphdb.repo_name,
    config.named_graphs.facts,
    frag_sn_states_facts_named_graph_name,
    frag_sn_states_events_facts_named_graph_name,
    frag_source_label
)

## Display evaluation results

### Results from first evaluation

In [ ]:
esv.print_version_quality_metrics(version_quality_metrics)

### Results from second evaluation

In [ ]:
esf.print_evaluation_tables(fragmentary_evaluation_metrics)